In [1]:
# stain_utils.py
import numpy as np

class MacenkoNormalizer:
    def __init__(self):
        # Calibrated H&E reference vectors
        self.HERef = np.array([
            [0.5626, 0.2159],
            [0.7201, 0.8012],
            [0.4062, 0.5581]
        ])
        self.maxCRef = np.array([1.9705, 1.0308])

    def normalize(self, img_rgb, Io=240, alpha=1, beta=0.15):
        h, w, c = img_rgb.shape
        img = img_rgb.reshape((-1, 3)).astype(float)

        # 1. Optical Density (OD) calculation
        OD = -np.log((img + 1.0) / Io)
        ODhat = OD[~np.any(OD < beta, axis=1)]
        if len(ODhat) == 0:
            return img_rgb  # Fallback for empty/blank areas

        # 2. Covariance eigenvectors
        _, eigvecs = np.linalg.eigh(np.cov(ODhat, rowvar=False))
        That = ODhat.dot(eigvecs[:, 1:3])

        phi = np.arctan2(That[:, 1], That[:, 0])
        minPhi = np.percentile(phi, alpha)
        maxPhi = np.percentile(phi, 100 - alpha)

        vMin = eigvecs[:, 1:3].dot(np.array([(np.cos(minPhi)), (np.sin(minPhi))]))
        vMax = eigvecs[:, 1:3].dot(np.array([(np.cos(maxPhi)), (np.sin(maxPhi))]))

        if vMin[0] > vMax[0]:
            HE = np.array((vMin, vMax)).T
        else:
            HE = np.array((vMax, vMin)).T

        # 3. Concentrations & reference scaling
        Y = np.reshape(OD, (-1, 3)).T
        C = np.linalg.lstsq(HE, Y, rcond=None)[0]
        maxC = np.percentile(C, 99, axis=1)

        C = C / maxC[:, None]
        C = C * self.maxCRef[:, None]

        # 4. Reconstruct normalized RGB
        norm_img = Io * np.exp(-self.HERef.dot(C))
        norm_img = np.clip(norm_img.T, 0, 255).astype(np.uint8)
        return norm_img.reshape((h, w, c))